# Sign Detection - Model Training (YOLOv8)

Notebook ini berisi alur lengkap untuk melatih model deteksi rambu 4 kelas (u_turn, stop, turn_left, turn_right) menggunakan Ultralytics YOLOv8 Nano.

In [ ]:
%pip install -q ultralytics onnx onnxslim
import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
import os
import sys
import yaml

# Auto-clone repository if running in Google Colab and dataset does not exist
if not os.path.exists("dataset") and not os.path.exists("../dataset"):
    !git clone https://github.com/IRIS-ITS/Nirvana-Sign-Detection.git
    %cd Nirvana-Sign-Detection

# Resolve absolute path to data.yaml and dataset directory
dataset_dir = os.path.abspath("../dataset")
if not os.path.exists(dataset_dir):
    dataset_dir = os.path.abspath("dataset")

yaml_path = os.path.join(dataset_dir, "data.yaml")

with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Ensure absolute path in data.yaml to prevent Ultralytics datasets_dir resolution conflicts
data_config["path"] = dataset_dir
with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"Data Config Path: {yaml_path}")
print("Dataset Configuration:")
print(data_config)

In [ ]:
# Load pre-trained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

In [ ]:
# Train YOLOv8n model (Phase 4 config: full-board labels + far/medium scale diversity)
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    scale=0.5,
    fliplr=0.5,
    degrees=15.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    close_mosaic=10,
    name="sign_detection",
    project="../runs/detect"
)

In [ ]:
# Evaluate model performance
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")

In [ ]:
# Export trained model weights to ONNX format for C++ OpenCV DNN
onnx_path = model.export(format="onnx", imgsz=640, opset=12, simplify=True)
print(f"Exported ONNX model to: {onnx_path}")

# Run automatic patcher script for OpenCV 4.5.4 C++ DNN compatibility
import subprocess
subprocess.run(["python3", "scripts/fix_onnx_opencv.py", onnx_path])

# Auto-download ONNX model file if running in Google Colab
try:
    from google.colab import files
    files.download(onnx_path)
    print("Downloading best.onnx to your local computer...")
except ImportError:
    print(f"Model saved locally at: {onnx_path}")